# 03 — Bell tests and teleportation: entanglement is not secret agreement

## What you will learn

Notebook 02 ended with an objection, and this notebook is the answer to it.

By the end you will know:

- what a **local hidden-variable** model is — the "pair of gloves" story — and why it
  was a perfectly reasonable thing to believe;
- the **CHSH game**, its score $S$, and the proof that no local hidden-variable model
  can score above $2$ — a brute-force search over all sixteen strategies, plus a
  one-line piece of algebra that explains the search's answer;
- how to **measure along a tilted axis**, which is the one genuinely new piece of
  physics here, and turns out to need no new machinery at all;
- the quantum strategy, which scores $2\sqrt2 \approx 2.828$, and what that looks like
  when estimated from actual sampled measurements rather than from the amplitudes;
- **what does and does not travel**: the no-signalling theorem, demonstrated
  numerically, and a careful account of what Bell's theorem actually rules out —
  including the reading in which nothing happens to the far particle at all;
- **teleportation**: moving an unknown quantum state using a shared Bell pair and two
  classical bits, walked through by hand one gate at a time;
- **superdense coding**, the same trade run in the opposite direction.

In notebook 02 we made a Bell pair and found that its two qubits always agree: measure
one, and the other gives the same answer, every time, however far apart they are. That
is strange, but there is a completely ordinary explanation sitting right there, and it
would be dishonest to skip past it.

Take a pair of gloves, put them in two boxes, and mail one to Tokyo. Open the box in
London, find the left glove, and you know instantly — faster than any signal could
travel — that the Tokyo box holds the right one. Nothing spooky happened. The gloves
were always what they were; you simply did not know which was which. If the qubits
agreed on their answers before they were separated, the Bell pair is a pair of gloves
and there is nothing more to say.

This is called a *local hidden-variable* model, and for thirty years it was a
respectable position. What makes it interesting is that it is not a matter of taste: in
1964 John Bell found a way to *test* it. There is a number you can measure that no
glove-like explanation can push above 2, and quantum mechanics predicts 2.83. This
notebook computes both.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsim import Circuit, Qubit
from qsim.algorithms.chsh import (
    CLASSICAL_LIMIT,
    OPTIMAL_SETTINGS,
    QUANTUM_LIMIT,
    chsh_expectation,
    chsh_S,
    chsh_sampled,
    classical_bound,
    classical_strategies,
)
from qsim.algorithms.teleportation import superdense_send, teleport
from qsim.gates import CNOT, H, Ry, Rz, X, Z

np.set_printoptions(precision=3, suppress=True)

# Every seed in this notebook comes from one master generator. Consecutive integers
# make correlated PCG64 streams, so `seed=1, 2, 3, ...` is not the same as
# "independent runs" — drawing the seeds from a generator is.
seed_rng = np.random.default_rng(20260801)


def seeds(n: int) -> list[int]:
    """n independent seeds, drawn from the master generator."""
    return [int(s) for s in seed_rng.integers(0, 2**32, size=n)]


def win_rate(s: float) -> float:
    """The CHSH win probability corresponding to a score S. Derived in section 1."""
    return 0.5 + s / 8

## 1. The CHSH game

Bell's test is easiest to meet as a **game**, in the form sharpened by Clauser, Horne,
Shimony and Holt in 1969 — hence *CHSH*.

There are three people. A **referee**, and two players, **Alice** and **Bob**, who are
put in separate rooms and cannot communicate in any way once the game starts. Each
round goes like this:

1. The referee flips two fair coins and hands Alice a question bit $x \in \{0, 1\}$ and
   Bob a question bit $y \in \{0, 1\}$. Neither player learns the other's question.
2. Alice answers $A = \pm 1$ and Bob answers $B = \pm 1$.
3. They win the round if their answers satisfy the condition in the table below.

| Alice's question $x$ | Bob's question $y$ | they win if |
|---|---|---|
| 0 | 0 | $A \cdot B = +1$ (answers **agree**) |
| 0 | 1 | $A \cdot B = +1$ (answers **agree**) |
| 1 | 0 | $A \cdot B = +1$ (answers **agree**) |
| 1 | 1 | $A \cdot B = -1$ (answers **differ**) |

So: agree in three of the four cases, differ in the fourth. The players know the rules
in advance and may plan together for as long as they like beforehand. They may carry
shared notes, shared dice, shared physical objects prepared together — anything at all,
as long as no information passes between the rooms *during* a round.

The single asymmetric cell is the whole game. If the players could agree "always answer
$+1$" they would win the first three cases and lose the fourth, for 75%. The question is
whether anything does better.

### From wins to a score

Counting wins works, but the standard bookkeeping uses **correlators**, which are easier
to compute and easier to compare with a physical prediction. For each pair of questions,
the correlator is the average of the product of the two answers:

$$E(x, y) = \big\langle A \cdot B \big\rangle \quad \text{over many rounds with those questions.}$$

Since $A$ and $B$ are each $\pm 1$, $E$ runs from $-1$ (they always differ) through $0$
(no correlation) to $+1$ (they always agree). The CHSH score bundles the four question
pairs into one number, with a minus sign on exactly the case whose win condition was
flipped:

$$S = E(0,0) + E(0,1) + E(1,0) - E(1,1).$$

The minus sign is doing the same job the table's last row does: for the three "agree"
cases a large positive $E$ is good, and for the "differ" case a large *negative* $E$ is
good, so subtracting it makes all four contribute positively when the players are doing
well.

Score and win rate carry exactly the same information. For the three "agree" rows the
win probability is $(1 + E)/2$ and for the last row it is $(1 - E)/2$; averaging the
four with equal weight gives

$$P(\text{win}) = \tfrac12 + \tfrac{S}{8}.$$

That is what `win_rate` above computes. $S = 2$ means 75%; $S = 2\sqrt2$ means 85.4%.
From here on we track $S$, because $S$ is where the theorem lives.

## 2. The best classical strategy

Now suppose the glove story is right — that whatever Alice and Bob carry into their
rooms, their answers were fixed in advance by it. Then the strategy space is small
enough to search exhaustively, and that is worth doing carefully, because "we tried hard
and could not beat 2" is a much weaker statement than "there is nothing else to try".

Here is why the space is small. A **deterministic local strategy** is a rule that turns
a question into an answer, separately on each side. Alice's rule has to specify only two
things: what she answers to $x = 0$ (call it $A$) and what she answers to $x = 1$ (call
it $A'$). Same for Bob: $B$ and $B'$. Each is $\pm 1$, so there are $2^4 = 16$
deterministic strategies in total, and for each one the four correlators are not
averages at all — they are just products, since nothing is random:

$$S = A B + A B' + A' B - A' B'.$$

What about *randomness*? A strategy that uses shared dice is a probability distribution
over deterministic strategies — the dice pick one, then both players follow it. $S$ is
linear in those probabilities, so the average of $S$ over a mixture can never exceed the
largest $S$ in the mixture. **Randomness cannot help.** That is why enumerating the 16
deterministic strategies settles the question for every local hidden-variable model, not
just the simple ones. (This is also where the word "hidden" earns its keep: the shared
variable may be arbitrarily complicated and completely unobservable. All that matters is
that it fixes the answers, and that neither answer depends on the *other* side's
question.)

`classical_strategies()` returns all sixteen rows with their scores.

In [ ]:
print(f"{'A':>3}{'A′':>4}{'B':>4}{'B′':>4}     {'S':>3}")
print("-" * 25)
for (a, a_prime, b, b_prime), s in classical_strategies():
    marker = "   <-- ties the record" if s == 2 else ""
    print(f"{a:>+3}{a_prime:>+4}{b:>+4}{b_prime:>+4}     {s:>+3}{marker}")

print()
print("best S over all 16 deterministic strategies:", classical_bound())
print("that is a win rate of:                      ", f"{win_rate(classical_bound()):.1%}")

Two things in that table deserve attention.

First, the maximum is exactly $2$, and eight of the sixteen strategies reach it. There is
no clever corner of the strategy space; half of the perfectly obvious strategies are
already optimal, and the other half score $-2$.

Second — and this is the striking part — **no strategy scores anything other than $\pm2$.**
Not $1$, not $1.7$, not $2.5$. The scores are quantized to two values, which is a strong
hint that something structural is going on rather than an arithmetic accident. Here is
the structure, in one line. Group the terms so that Bob's answers appear together:

$$S = A B + A B' + A' B - A' B' = A\,(B + B') + A'\,(B - B').$$

$B$ and $B'$ are each $\pm 1$, so there are only two cases:

- **they are equal.** Then $B - B' = 0$, killing the second term, and $B + B' = \pm 2$.
- **they differ.** Then $B + B' = 0$, killing the first term, and $B - B' = \pm 2$.

Either way exactly one term survives, and it is $(\pm 1) \times (\pm 2) = \pm 2$. There
is nothing to optimize; the answer was never going to be anything else.

That argument used no physics whatsoever. It used only the assumption that $A$, $A'$,
$B$, $B'$ are four numbers that simultaneously exist — that Alice's answer to the
question she was *not* asked is still a well-defined $\pm 1$ sitting in the algebra. Hold
on to that, because it is the assumption quantum mechanics declines to make.

> **The Bell/CHSH inequality.** For any local hidden-variable theory, $|S| \le 2$.

It is worth pausing on how unusual this is. It is a limit on what *any* theory of a
certain broad shape can predict, derived without knowing anything about the theory. Bell
turned a philosophical dispute into an experiment.

## 3. Measuring along a tilted axis

To beat 2 the players need to give Alice and Bob *different questions* in a sense
stronger than "different bits" — the two questions have to be genuinely incompatible, in
the way that $Z$ and $X$ measurements on a single qubit are. So far every measurement in
these notebooks has asked one question: "is this qubit $|0\rangle$ or $|1\rangle$?" That
is a measurement along the $z$-axis of the Bloch sphere. We now need axes in between.

The observable for the axis tilted by an angle $\theta$ from $z$ towards $x$ is

$$M_\theta = \cos\theta \cdot Z + \sin\theta \cdot X,$$

which is a Hermitian $2\times2$ matrix with eigenvalues $\pm 1$, just like $Z$ and $X$
themselves. At $\theta = 0$ it is $Z$; at $\theta = \pi/2$ it is $X$. Its two eigenvectors
are the two points where the tilted axis pierces the Bloch sphere, and measuring it means
asking "which of those two are you?" — with the usual Born-rule probabilities and the
usual collapse.

Here is the pleasant part: **this needs no new machinery.** Rotating the *apparatus* by
$\theta$ and rotating the *sample* by $-\theta$ are the same experiment. So

$$\text{apply } R_y(-\theta), \text{ then measure } Z \quad = \quad \text{measure } M_\theta .$$

$R_y$ is the rotation about the $y$-axis of the Bloch sphere that notebook 01 introduced,
and it is the right one because rotating about $y$ is what tips $z$ towards $x$. The
minus sign is the "turn the sample instead of the apparatus" bookkeeping: to ask a
question tilted *forwards* by $\theta$, tip the state *backwards* by $\theta$ and ask the
question you already know how to ask.

That is a claim, so let us check it numerically on an arbitrary state.

In [ ]:
def some_state(seed: int = 0) -> tuple[Circuit, Qubit]:
    """An arbitrary, deliberately unspecial single-qubit state."""
    c = Circuit(name="tilt", seed=seed)
    q = c.alloc()
    Ry(q, theta=0.9)
    Rz(q, theta=0.4)
    return c, q


print(f"{'θ':>6}   {'cosθ·⟨Z⟩ + sinθ·⟨X⟩':>21}   {'⟨Z⟩ after Ry(-θ)':>21}")
for theta in [0.0, np.pi / 6, np.pi / 4, 1.3, np.pi / 2]:
    direct, _ = some_state()
    # <M_theta> expanded by linearity: the expectation of a sum of observables is the
    # sum of their expectations, weighted the same way.
    tilted = np.cos(theta) * direct.inspect.expectation("Z")
    tilted += np.sin(theta) * direct.inspect.expectation("X")

    turned, q = some_state()
    Ry(q, theta=-theta)  # turn the sample instead of the apparatus
    print(f"{theta:>6.3f}   {tilted:>21.15f}   {turned.inspect.expectation('Z'):>21.15f}")

The two columns agree to fourteen decimal places at every angle — where the last digit
differs, that is double-precision rounding, not physics. "Measure at an angle" is one
extra gate, and every measurement in the rest of this notebook is still an ordinary
computational-basis measurement.

Now put it on a Bell pair. Alice rotates her half by $-\alpha$ and Bob rotates his by
$-\beta$, and then both measure $Z$; the correlator $E(\alpha, \beta)$ is the average of
the product of their $\pm1$ answers. `chsh_expectation` does exactly this, reading
$\langle ZZ\rangle$ off the amplitudes — a shortcut no real experiment has, which is why
`chsh_sampled` exists too and why we will use it before claiming anything.

The prediction for a Bell pair is
$$E(\alpha, \beta) = \cos(\alpha - \beta),$$
which depends only on the *angle between* the two apparatus settings, not on either
setting separately. Same axis, perfect agreement; perpendicular axes, no correlation at
all; and a smooth cosine in between.

In [ ]:
print("  α        β        E(α,β)      cos(α−β)")
for alpha, beta in [(0.0, 0.0), (0.0, np.pi / 4), (0.0, np.pi / 2),
                    (np.pi / 2, np.pi / 4), (0.0, np.pi)]:
    e = chsh_expectation(alpha, beta)
    print(f"{alpha:7.4f}  {beta:7.4f}   {e:+.9f}   {np.cos(alpha - beta):+.9f}")

### The quantum strategy

Now the players' plan. They share a Bell pair, one qubit each, prepared together before
the game and then separated. Each of them fixes *two* measurement angles in advance, one
per possible question:

- Alice measures at $a = 0$ if her question is $0$, and at $a' = \pi/2$ if it is $1$;
- Bob measures at $b = \pi/4$ if his question is $0$, and at $b' = -\pi/4$ if it is $1$.

Each answers with the outcome of that measurement, $+1$ for $|0\rangle$ and $-1$ for
$|1\rangle$. Note what has *not* changed: neither player learns the other's question, and
nothing is transmitted. All that has changed is the physical object they carry into their
rooms.

Since $E(\alpha, \beta) = \cos(\alpha - \beta)$, the arithmetic is short. The three
correlators that get added involve angle differences of $\pm\pi/4$, each contributing
$\cos(\pi/4) = 1/\sqrt2$. The one that gets subtracted has an angle difference of
$3\pi/4$, so $\cos(3\pi/4) = -1/\sqrt2$, and subtracting it *adds* another $1/\sqrt2$:

$$S = \tfrac{1}{\sqrt2} + \tfrac{1}{\sqrt2} + \tfrac{1}{\sqrt2} - \left(-\tfrac{1}{\sqrt2}\right)
    = \tfrac{4}{\sqrt2} = 2\sqrt2 \approx 2.828.$$

The clever bit is entirely in the choice of angles: Bob's two settings straddle Alice's
two, so all four pairings are $45°$ apart, and the sign structure of $S$ converts the one
"bad" pairing into a fourth good one.

In [ ]:
s_exact = chsh_S()
s_shots = chsh_sampled(shots=100_000, seed=seeds(1)[0])

print("optimal settings (a, a′, b, b′) =", np.array(OPTIMAL_SETTINGS))
print()
print(f"classical ceiling            S = {classical_bound():.6f}"
      f"    win rate {win_rate(classical_bound()):6.2%}")
print(f"quantum, from the amplitudes S = {s_exact:.6f}"
      f"    win rate {win_rate(s_exact):6.2%}")
print(f"quantum, from 100000 shots   S = {s_shots:.6f}"
      f"    win rate {win_rate(s_shots):6.2%}")
print()
print(f"Tsirelson's bound 2√2        = {QUANTUM_LIMIT:.6f}")
print(f"exceeds the classical limit of {CLASSICAL_LIMIT} by {s_exact - CLASSICAL_LIMIT:.4f}")

$2.828$, from a strategy that transmits nothing.

The third line is the one that matters, because the first two were computed by reading
amplitudes off a state vector — something no laboratory can do. `chsh_sampled` instead
draws $100{,}000$ actual $\pm1$ outcomes per correlator and averages them, which is what
an experiment reports. It lands near $2\sqrt2$ rather than on it, by a few parts in a
thousand, because $100{,}000$ samples give an uncertainty of order $1/\sqrt{100000}
\approx 0.003$ per correlator. The violation has to be visible through that noise, and it
is: the gap to the classical ceiling is $0.83$, hundreds of standard deviations wide.

Two boundaries are now on the table, and they are different in kind:

- $S \le 2$ is the **classical limit**, and it came from counting. Any theory in which
  the answers were fixed in advance obeys it.
- $S \le 2\sqrt2$ is **Tsirelson's bound**, and it is quantum mechanics' own ceiling.
  Nothing in the formalism reaches higher. Curiously, no-signalling *alone* would permit
  scores up to $4$ — one can write down consistent toy theories that hit it — so quantum
  mechanics is not simply "as non-local as it could be without breaking relativity". It
  sits at a specific point in between, and why *that* point is still an open question.

This is not a thought experiment. Versions of it have been run in laboratories since the
1970s, with each successive experiment closing another way the result could be faked:
detectors chosen too slowly, particles lost before detection, settings not chosen
independently enough. The 2015 "loophole-free" experiments closed the main ones
simultaneously, and the 2022 Nobel Prize in Physics went to **Alain Aspect, John Clauser
and Anton Zeilinger** for that programme of work. The number in the third line above is
measured, not merely calculated.

## 4. The violation is a band, not a knife edge

A fair worry at this point: the quantum strategy used four very specific angles, and
$2\sqrt2$ is the exact maximum. Does the whole effect depend on hitting them precisely?
If it did, a real experiment — where angles have error bars — would be in trouble.

Take Alice's two angles as fixed and slide *both* of Bob's by the same offset $\delta$,
so his settings become $\pi/4 + \delta$ and $-\pi/4 + \delta$. Then plot $S$ against
$\delta$, and overlay a handful of sampled runs, each using only $2{,}000$ shots per
correlator so the statistical scatter is actually visible.

In [ ]:
def bob_offset(delta: float) -> tuple[float, float, float, float]:
    """Optimal settings with both of Bob's angles slid by delta."""
    return (0.0, np.pi / 2, np.pi / 4 + delta, -np.pi / 4 + delta)


offsets = np.linspace(-np.pi / 2, np.pi / 2, 201)
curve = [chsh_S(bob_offset(d)) for d in offsets]

shot_offsets = [-1.3, -0.7, -0.2, 0.2, 0.7, 1.3]
shot_scores = [chsh_sampled(shots=2000, seed=s, settings=bob_offset(d))
               for d, s in zip(shot_offsets, seeds(len(shot_offsets)))]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(offsets, curve, linewidth=2, label="exact S (read off the amplitudes)")
ax.plot(shot_offsets, shot_scores, "o", color="crimson", zorder=3,
        label="S from 2000 shots per correlator")
ax.axhline(CLASSICAL_LIMIT, color="gray", linestyle="--", linewidth=1.3)
ax.text(-1.55, 2.06, "classical limit  S = 2", color="gray", fontsize=9)
ax.axhline(QUANTUM_LIMIT, color="darkgreen", linestyle=":", linewidth=1.3)
ax.text(-1.55, QUANTUM_LIMIT + 0.06, "Tsirelson bound  S = 2√2", color="darkgreen", fontsize=9)
ax.axvspan(-np.pi / 4, np.pi / 4, color="tab:blue", alpha=0.08)
ax.set_xlabel("offset δ added to both of Bob's angles (radians)")
ax.set_ylabel("CHSH score S")
ax.set_title("CHSH score as Bob's apparatus is rotated away from the optimum")
ax.set_ylim(0, 3.15)
ax.legend(loc="lower center", fontsize=9)

print("S at δ = 0        :", round(chsh_S(bob_offset(0.0)), 6))
print("S at δ = ±π/4     :", round(chsh_S(bob_offset(np.pi / 4)), 6))
print("S at δ = ±π/2     :", round(chsh_S(bob_offset(np.pi / 2)), 6))

The curve is $2\sqrt2 \cos\delta$: a single smooth cosine, peaking at $2\sqrt2$ when
Bob's apparatus is aligned as planned and falling to $0$ when he has rotated a full
$90°$ away.

The shaded band is where $S > 2$. It runs from $\delta = -\pi/4$ to $\delta = +\pi/4$,
because $2\sqrt2\cos\delta > 2$ exactly when $\cos\delta > 1/\sqrt2$. Bob can be
**$45°$ out of alignment** and the inequality is still violated. This is a robust
effect, not a resonance, which is a large part of why the experiments were feasible at
all.

The red dots are sampled runs with only $2{,}000$ shots per correlator. They scatter
around the curve by a couple of hundredths — visible, but nowhere near enough to bridge
the gap between $2.83$ and $2$. Noise blurs the answer; it does not manufacture it.

## 5. But what actually travelled?

$S = 2.83$ rules out the glove story. It is very tempting to conclude that Alice's
measurement must therefore *do something* to Bob's particle — reach across the gap and
set it. That conclusion is stronger than the evidence, and this section is about being
precise, because this is exactly the point where popular accounts go wrong.

Start with what can be checked by computation rather than argued about.

### Alice sees a fair coin, no matter what Bob does

Take a Bell pair. Bob picks a measurement angle — that is, he applies $R_y(-\theta)$ to
his own qubit and to nothing else. Ask what Alice's qubit looks like, using the reduced
density matrix from notebook 02.

In [ ]:
def bell_pair(seed: int) -> tuple[Circuit, Qubit, Qubit]:
    """A fresh (|00> + |11>)/sqrt(2), with handles to Alice's half and Bob's half."""
    c = Circuit(name="bell", seed=seed)
    a, b = c.alloc_many(2)
    H(a)
    CNOT(a, b)
    return c, a, b


fixed_seed = seeds(1)[0]
print("Alice's reduced density matrix as Bob turns his apparatus (without measuring):")
for theta in [0.0, 0.3, np.pi / 4, 1.9, np.pi]:
    c, a, b = bell_pair(fixed_seed)
    Ry(b, theta=-theta)
    rho = c.inspect.reduced_density_matrix([a])
    # Round away the 1e-17 dust so the pattern is readable; .real is safe here only
    # because we check the imaginary part is negligible first.
    assert np.max(np.abs(rho.imag)) < 1e-12
    print(f"  Bob's angle {theta:5.3f}  ->  {np.round(rho.real, 12).tolist()}")

Exactly $\tfrac12 I$ every time — the maximally mixed state, a fair coin, identical to
twelve decimal places whatever Bob chose. Bob's *choice of setting* is completely
invisible on Alice's side.

That one is easy to prove in general, and the proof is one line of the linear algebra
you already have. If Bob applies a unitary $U$ to his qubit alone, the joint state
becomes $(I \otimes U)\rho(I \otimes U^\dagger)$, and tracing out Bob's side gives
$\operatorname{Tr}_B\!\big[(I \otimes U)\rho(I \otimes U^\dagger)\big] =
\operatorname{Tr}_B[\rho]$, because the trace is cyclic and $U^\dagger U = I$. Alice's
reduced state is untouched. Nothing Bob does to his own tensor factor can change it.

### What about when Bob actually measures?

Here we have to be careful, because the honest answer has two parts and only quoting the
second one is how people end up believing something false.

If Bob measures and you **condition on his result**, Alice's state does change — that is
collapse, and it is not subtle:

In [ ]:
c, a, b = bell_pair(fixed_seed)
Ry(b, theta=-np.pi / 4)
outcome = c.measure(b)
print(f"Bob measured at π/4 and got {outcome}.")
print("Alice's state *given that result*:")
print(np.round(c.inspect.reduced_density_matrix([a]).real, 3))
print("its Bloch vector length:", round(float(np.linalg.norm(c.inspect.bloch_vector(a))), 6))

A Bloch vector of length $1$: Alice's qubit is now in a definite pure state, no longer a
fair coin. If Alice could see *that*, Bob could signal to her instantly.

She cannot, and here is the second part. Alice has no access to Bob's outcome, so she
cannot sort her runs into his two piles. What she has is the whole ensemble: every run,
mixed together in the proportions the Born rule gives. Average over Bob's outcomes and
the change cancels exactly.

In [ ]:
n_runs = 2000
run_seeds = seeds(n_runs)

print(f"Alice's density matrix averaged over {n_runs} runs in which Bob measured:")
for theta in [0.0, np.pi / 4, 1.9]:
    total = np.zeros((2, 2), dtype=complex)
    for s in run_seeds:
        c, a, b = bell_pair(s)
        Ry(b, theta=-theta)
        c.measure(b)  # Bob measures; Alice is never told the answer
        total += c.inspect.reduced_density_matrix([a])
    print(f"  Bob's angle {theta:5.3f}  ->  {np.round((total / n_runs).real, 3).tolist()}")

print()
print("(the exact answer is [[0.5, 0], [0, 0.5]]; the wobble is 1/sqrt(2000) ≈ 0.02)")

Back to $\tfrac12 I$, up to sampling noise, at every angle. And again there is a
one-line proof behind the numbers: averaging over Bob's outcomes means forming
$\sum_m (I \otimes P_m)\,\rho\,(I \otimes P_m)$ with $P_m$ his projectors, and tracing
out Bob's side gives $\operatorname{Tr}_B[\rho]$ again, since $\sum_m P_m^\dagger P_m = I$.

> **The no-signalling theorem.** No operation Bob performs on his half of an entangled
> pair — any unitary, any measurement, in any basis, at any time — changes the
> statistics Alice observes on hers. Entanglement cannot be used to send a message.

So the violation is not something either party can *see*. Alice has a list of $\pm1$s
that looks like coin flips. Bob has a list of $\pm1$s that looks like coin flips. $S$
only appears when the two lists are laid side by side and compared row by row — and
getting them into the same room requires an ordinary message, travelling at or below
the speed of light, like everything else. Quantum mechanics and relativity are not in
conflict here; the tension is entirely in the story we tell about the mechanism.

### So what does Bell actually rule out?

The theorem is usually reported as "quantum mechanics is non-local", and that is a
compression of something more careful. Deriving $|S| \le 2$ took **three** assumptions,
not one:

**(a) Locality.** No influence travels faster than light. Alice's outcome does not depend
on Bob's setting, and vice versa.

**(b) Definite single outcomes.** Each measurement had one actual result. This is what
let us write $A$, $A'$, $B$, $B'$ as four ordinary numbers in the same expression in
section 2 — including the answers to the questions that were *not* asked.

**(c) Settings independence.** Alice and Bob choose their angles freely, by a process
uncorrelated with whatever the particles carry. (Also called measurement independence or
"freedom of choice". This is why later experiments chose settings from distant quasar
light: to make any correlation between choice and particle implausibly old.)

$S > 2$ means **at least one of the three has to go.** The theorem does not, by itself,
say which — and that choice is what different interpretations of quantum mechanics
disagree about. Give up (a) and you get explicitly non-local hidden variables, such as
Bohmian mechanics. Give up (c) and you get superdeterminism, which almost nobody likes.
Give up (b) and you get the reading in the next subsection.

Different presentations of Bell's theorem slice the package slightly differently — some
combine (a) and (b) into a single "local realism" — but the count is the point: the
experiment refutes a *conjunction*, and a refuted conjunction does not tell you which
conjunct was wrong.

### The observer-centred reading

Here is a thought that occurs naturally to anyone who takes the no-signalling result
seriously. Perhaps nothing happens to the far particle at all. Perhaps measuring the
near particle changes *me* — puts the observer into a state such that, when they later
encounter the second particle, they encounter it "the right way".

This is a serious position, and it is close to the **relative-state** reading of quantum
mechanics that Hugh Everett proposed in 1957 (usually now called "many-worlds", which is
a worse name for it). Bell's theorem does **not** rule it out, because it gives up
assumption (b) rather than assumption (a). If there is no single outcome to Alice's
measurement, then $A$ and $A'$ are not four numbers sitting in one algebraic expression,
the derivation in section 2 never starts, and no locality has to be sacrificed.

The price is specific, and worth stating plainly rather than glossing. In this reading
the measurement is an ordinary unitary interaction: the observer *entangles* with the
particle, exactly as the CNOT in `bell_pair` entangles two qubits. So after measuring,
the observer is not in a definite state either. They are in a superposition of "saw
$+1$" and "saw $-1$", and by the argument of notebook 02 section 3 they have no state of
their own at all — only a relationship with the particle. There is a definite record
*relative to each branch*, but no branch-independent fact about what was seen, and the
correlation between Alice's branch and Bob's branch only becomes a fact when the two
records meet and entangle with each other. That is not a loophole in Bell's argument. It
is a different, and considerably larger, bill.

### The trap to avoid

There is a version of this thought that sounds the same and is definitely wrong, so it is
worth separating them.

If the change to the observer is an ordinary **classical** fact — a definite record, one
outcome, that happens to be correlated with values the particles were already carrying —
then that is a local hidden-variable model, however it is dressed up. The record is the
hidden variable. Section 2's ceiling of $2$ applies to it directly, and the experiments
say $2.83$. The observer-centred reading survives Bell **only** if the observer's state
after measuring is genuinely quantum: superposed, not merely unknown. "I have a definite
result and just haven't compared notes yet" is the glove story with an extra step.

That distinction — superposition versus mere ignorance — is exactly the distinction
between a density matrix with coherences and one without, which notebook 02 built the
machinery for. It is not a matter of words.

### Where this notebook stops, and notebook 05 picks up

None of this is idle philosophy for this project, because "the observer becomes entangled
with what they measure" is not just an interpretive slogan. It is **decoherence**, and
notebook 05 builds it explicitly: a system coupled unitarily to an environment, no
collapse postulate anywhere, and the appearance of a definite classical outcome emerging
from the entanglement alone. You will watch a measurement's apparent collapse produced by
entangling gates — and then, in the quantum eraser, watch it *undone*, which is only
possible because the environment was kept rather than discarded. That is the strongest
handle this project can give you on the question, and it is a computation rather than an
argument.

Two honest caveats to close on.

First, these readings are **interpretations**. As far as anyone knows they predict
identical results for every experiment that has been done or proposed. This is a question
about what the mathematics *means*, not about what a machine will print.

Second, notice what this section has *not* claimed. Not that anything travelled. Not that
Alice's measurement changed Bob's particle. Not "spooky action at a distance" — Einstein's
phrase, coined as a complaint about a theory he thought incomplete, and not a description
of a mechanism. The defensible claim is exactly the one CHSH licenses, and it is quite
strong enough:

> No theory in which (a) influences are local, (b) measurements have single definite
> outcomes, and (c) settings are chosen independently can reproduce what quantum
> mechanics predicts and what experiments measure.

## 6. Teleportation

Entanglement cannot send a message on its own. Paired with an ordinary message, though,
it can do something ordinary messages cannot do alone: move a quantum state.

Set up the problem first, because the obstacle is what makes the protocol interesting.
Alice holds a qubit in some state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ that
she does **not** know — perhaps it came out of a computation, perhaps someone handed it
to her. She wants Bob, far away, to end up with a qubit in that state. She has a
telephone, and no way to ship a qubit.

The obvious approaches all fail, and they fail for reasons this course has already
established:

- **Measure it and describe the result.** Measuring returns one bit, chosen randomly,
  and destroys the superposition that held everything else. $\alpha$ and $\beta$ are two
  continuous complex numbers; one bit does not locate them.
- **Copy it first, then experiment on the copy.** Forbidden. That is the no-cloning
  theorem from notebook 02, which is why `copy.copy(qubit)` raises `NoCloningError`.
- **Measure many copies to estimate $\alpha$ and $\beta$.** She has one qubit, and cannot
  make more.

So the state cannot be learned, and it cannot be duplicated. Teleportation moves it
anyway, without Alice ever learning what she sent.

### The protocol

Three qubits, and one Bell pair distributed in advance — before there was any message.

| qubit | who holds it | starts as |
|---|---|---|
| `msg` | Alice | the unknown state $|\psi\rangle$ |
| `a` | Alice | half of a Bell pair |
| `b` | Bob (far away) | the other half |

1. **Alice entangles the message with her half of the pair:** `CNOT(msg, a)`, then
   `H(msg)`. This is the Bell-pair recipe from notebook 02 run *backwards* — the same
   two gates in the opposite order — which turns Bell states into basis states instead of
   the other way round.
2. **Alice measures both of her qubits**, getting two classical bits $m_1$ and $m_2$.
   Both are uniformly random regardless of $|\psi\rangle$: she learns nothing about the
   state she is sending. This is also the moment the state disappears from her side.
3. **Alice telephones Bob the two bits.** Ordinary classical communication, at ordinary
   speed.
4. **Bob applies a correction:** `X(b)` if $m_2 = 1$, then `Z(b)` if $m_1 = 1$. His qubit
   is now in the state $|\psi\rangle$, exactly.

Step 3 is the whole reason this does not violate anything from section 5. Before the
call, Bob's qubit is maximally mixed and useless; the two bits are what make it usable,
and they crawl along a telephone line like everything else. And step 2 is the reason it
does not violate no-cloning: the original is *gone*, not copied.

Let us run it by hand, one gate at a time, printing the full three-qubit state at every
step. The state we will send is $R_y(1.1)|0\rangle \approx 0.853|0\rangle +
0.523|1\rangle$ — "unknown" in the sense that no step of the protocol will use those
numbers. The seed is fixed at 2 so the prose below matches the output.

In [ ]:
theta = 1.1
tp = Circuit(name="teleport by hand", seed=2)
msg, a, b = tp.alloc_many(3)

# Ordering of the three axes is (msg, a, b), so the ket labels read |msg a b>.
Ry(msg, theta=theta)
print("1. message prepared      ", tp.inspect.ket())

H(a)
CNOT(a, b)
print("2. Bell pair distributed ", tp.inspect.ket())

CNOT(msg, a)
H(msg)
print("3. Alice's two gates     ", tp.inspect.ket())

m1 = tp.measure(msg)
m2 = tp.measure(a)
print(f"4. Alice measures -> m1={m1}, m2={m2}")
print("   state after collapse  ", tp.inspect.ket())

# "Alice telephones Bob." Classical communication really is just an `if` on a bit.
if m2 == 1:
    X(b)
if m1 == 1:
    Z(b)
print("5. Bob's correction      ", tp.inspect.ket())

Read that trace line by line; every step earns its place.

**Line 1.** $0.853|000\rangle + 0.523|100\rangle$. Only the first qubit is doing
anything: the amplitudes $\cos(1.1/2)$ and $\sin(1.1/2)$ sit in front of the two values
of `msg`, with `a` and `b` still $|00\rangle$.

**Line 2.** Four terms. The message amplitudes have each split in two, once for
$|00\rangle$ of the pair and once for $|11\rangle$. Nothing has mixed yet — this is
literally $|\psi\rangle \otimes \tfrac{1}{\sqrt2}(|00\rangle + |11\rangle)$, a product of
Alice's unknown state with the shared pair.

**Line 3.** Eight terms, all with magnitude $0.426 = \alpha/2$ or $0.261 = \beta/2$, and
— crucially — two of them with minus signs. This is the state the whole protocol turns
on. Group it by what Alice's two qubits read:
$$\tfrac12\Big(|00\rangle\,|\psi\rangle + |01\rangle\,X|\psi\rangle
 + |10\rangle\,Z|\psi\rangle + |11\rangle\,XZ|\psi\rangle\Big),$$
where the first ket is Alice's pair and the second is Bob's qubit. You can read the four
branches straight off the printed line: the $|00\rangle$ branch has coefficients
$(+\alpha/2, +\beta/2)$, the $|01\rangle$ branch $(+\beta/2, +\alpha/2)$ — that is
$|\psi\rangle$ with its two amplitudes swapped, i.e. $X|\psi\rangle$ — and so on.

**Bob's qubit already holds the message in all four branches**, each one merely scrambled
by a different Pauli. Alice's two gates did not send anything anywhere; they rearranged
which question her measurement is going to ask.

**Line 4.** Alice measured $m_1 = 1, m_2 = 1$, selecting the fourth branch. Everything
else is gone and the state is $|11\rangle \otimes XZ|\psi\rangle$ — the message is sitting
on Bob's qubit, flipped and phase-flipped. Alice's own qubits now read a plain classical
$|11\rangle$: whatever they were carrying, they are not carrying it any more.

**Line 5.** Bob undoes the scrambling. He got the bits $1, 1$, so he applies $X$ and then
$Z$, which composes to the operator $ZX$; since $X$ and $Z$ are each their own inverse,
$ZX \cdot XZ = Z\,(XX)\,Z = ZZ = I$. The final state is
$|11\rangle \otimes (0.853|0\rangle + 0.523|1\rangle)$ — the original amplitudes, now on
Bob's qubit.

Note what Bob's correction did *not* require: any knowledge of $\alpha$ and $\beta$. He
applied two fixed gates chosen by two classical bits. Neither party ever learned the
state.

In [ ]:
reference = Circuit()
Ry(reference.alloc(), theta=theta)

# Print as plain lists: np.set_printoptions(precision=3) above would otherwise hide
# the digits we are trying to compare.
print("Bob's Bloch vector: ", [round(v, 9) for v in tp.inspect.bloch_vector(b)])
print("the target's:       ",
      [round(v, 9) for v in reference.inspect.bloch_vector(reference.qubits[0])])

print()
print("Alice's leftovers — Bloch z-components (±1 means a definite classical bit):")
print("  msg:", round(tp.inspect.bloch_vector(msg)[2], 9))
print("  a:  ", round(tp.inspect.bloch_vector(a)[2], 9))
print("  entanglement entropy of {msg, a}:", round(tp.inspect.entanglement_entropy([msg, a]), 12))

Bob's Bloch vector is the target's, to nine decimal places. Alice's two qubits sit at
Bloch $z = \pm1$ — the poles of the sphere, which is what "an ordinary classical bit"
looks like geometrically — with zero entanglement entropy between them and the rest.

That last measurement is where the no-cloning bookkeeping is actually checked. If any
trace of $|\psi\rangle$ were still on Alice's side, her qubits would be somewhere off the
poles, or still entangled with Bob's. They are not. One copy went in and one copy came
out; what moved was the state, not a duplicate of it.

### The packaged version, and all four branches

`teleport()` does all of the above and reports what happened. The branch is chosen by
Alice's random measurement outcomes, so varying the seed walks through all four.

In [ ]:
def prep_ry(angle: float):
    """Build a state_prep callable: put a fresh |0> qubit into Ry(angle)|0>."""
    def prep(q: Qubit) -> None:
        Ry(q, theta=angle)
    return prep


branches: dict[tuple[int, int], object] = {}
for s in seeds(80):
    r = teleport(prep_ry(1.1), seed=s)
    branches.setdefault((r.m1, r.m2), r)

print(f"{'m1 m2':^7}   {'Bob applies':>12}   {'fidelity':>18}   Alice's Bloch z")
for key in sorted(branches):
    r = branches[key]
    label = r.corrections if r.corrections else "(nothing)"
    z = ", ".join(f"{v:+.0f}" for v in r.source_bloch_z)
    print(f"{f'{key[0]}  {key[1]}':^7}   {label:>12}   {r.fidelity:>18.15f}   ({z})")

All four branches, fidelity $1$ in every one — to fifteen digits, with the last one or
two being ordinary floating-point dust. **Fidelity** here is
$\langle\psi|\rho_b|\psi\rangle$: the probability that a measurement asking "are you the
target state?" answers yes. A value of $1$ means the arrival is exact, not approximate.

The last column is the honest part of the ledger. Alice's two Bloch $z$-components are
$\pm1$ in every branch — always a definite classical bit, never a fraction. Compare the
two ends of the protocol: before, Alice had an unknown quantum state and two random bits'
worth of nothing; after, Bob has the unknown quantum state and Alice has two random bits.
The books balance exactly, which is what no-cloning demands.

Teleportation works for *any* input state, not just the one we picked. It never touches
$\alpha$ and $\beta$, so it cannot depend on them:

In [ ]:
angle_rng = np.random.default_rng(seeds(1)[0])


def prep_random(ry: float, rz: float):
    """A state_prep for an arbitrary point on the Bloch sphere."""
    def prep(q: Qubit) -> None:
        Ry(q, theta=ry)
        Rz(q, theta=rz)
    return prep


print(f"{'Ry angle':>9} {'Rz angle':>9}   {'m1 m2':>6}   fidelity")
for s in seeds(6):
    ry, rz = angle_rng.uniform(0, 2 * np.pi, size=2)
    r = teleport(prep_random(float(ry), float(rz)), seed=s)
    print(f"{ry:>9.4f} {rz:>9.4f}   {r.m1}  {r.m2:>2}   {r.fidelity:.15f}")

## 7. Superdense coding: the same trade, backwards

Teleportation spends **one Bell pair and two classical bits** to move **one qubit**.
Superdense coding runs the ledger the other way: spend **one Bell pair and one qubit** to
move **two classical bits**.

The mechanism is the mirror image too. Alice and Bob share a Bell pair, as before. Now
Alice wants to send two classical bits, and she does it by acting on her half alone:

| bits | Alice applies | resulting Bell state |
|---|---|---|
| `(0, 0)` | nothing | $\tfrac{1}{\sqrt2}(|00\rangle + |11\rangle)$ |
| `(0, 1)` | $X$ | $\tfrac{1}{\sqrt2}(|01\rangle + |10\rangle)$ |
| `(1, 0)` | $Z$ | $\tfrac{1}{\sqrt2}(|00\rangle - |11\rangle)$ |
| `(1, 1)` | $Z$ then $X$ | $\tfrac{1}{\sqrt2}(|01\rangle - |10\rangle)$ |

Those four states are the **Bell basis**: four mutually orthogonal states of two qubits,
so they are perfectly distinguishable from one another *if you hold both qubits*. Alice
then physically hands her one qubit to Bob, who now does hold both, and runs the
Bell-pair recipe backwards — `CNOT` then `H`, exactly Alice's step 1 in teleportation —
which maps the four Bell states onto the four basis states $|00\rangle, |01\rangle,
|10\rangle, |11\rangle$. He measures, and reads off two bits.

Keep the accounting honest: two qubits were involved in total. Bob simply received one of
them earlier, back when there was no message to send. The interesting claim is not "one
qubit carries two bits from nowhere" but "a Bell pair distributed in advance doubles the
capacity of a qubit sent later".

In [ ]:
print("message  ->  decoded   correct?")
for message, s in zip([(0, 0), (0, 1), (1, 0), (1, 1)], seeds(4)):
    received = superdense_send(message, seed=s)
    print(f" {message}   ->   {received}      {received == message}")

All four round-trip exactly, and they do so with certainty rather than on average — the
Bell states are orthogonal, so Bob's measurement has no chance of confusing them. Notice
the pattern of gates across the two protocols: `CNOT` then `H` appears in both, once to
*create* the correlation and once to *read* it. Teleportation and superdense coding are
the same circuit run in opposite directions, which is why they trade the same three
resources.

## What you now know

- A **local hidden-variable** model says the particles carried their answers all along.
  It is the pair-of-gloves story, it was a respectable position, and it is testable.
- In the **CHSH game** Alice and Bob answer $\pm1$ to random question bits without
  communicating. The score $S = E(0,0) + E(0,1) + E(1,0) - E(1,1)$ is related to the win
  rate by $P = \tfrac12 + \tfrac{S}{8}$.
- **Every** deterministic local strategy scores exactly $\pm2$, because
  $S = A(B + B') + A'(B - B')$ and one bracket always vanishes while the other is $\pm2$.
  Shared randomness cannot help, since $S$ is linear in the mixing probabilities. Hence
  $|S| \le 2$ for any local hidden-variable theory.
- **Measuring along a tilted axis** — the observable $\cos\theta\,Z + \sin\theta\,X$ —
  needs no new machinery: apply $R_y(-\theta)$ and measure $Z$. Turning the sample is the
  same experiment as turning the apparatus.
- A Bell pair gives $E(\alpha, \beta) = \cos(\alpha - \beta)$, and the settings
  $(0, \pi/2, \pi/4, -\pi/4)$ give $S = 2\sqrt2 \approx 2.828$, an 85.4% win rate. The
  violation survives up to $45°$ of misalignment and is visible through sampling noise.
  Real experiments measure it; the 2022 Nobel Prize was awarded for them.
- **No-signalling**: nothing Bob does to his half — any unitary, any measurement, any
  basis — changes Alice's statistics. Conditioning on his outcome changes her state; the
  average over his outcomes does not, and the average is all she has. $S$ becomes visible
  only when the two lists of results are compared, which needs an ordinary message.
- Bell's theorem refutes a **conjunction** of three assumptions — locality, definite
  single outcomes, settings independence — and does not say which one to drop. The
  observer-centred (relative-state) reading drops the second, at the price of the observer
  themselves being in superposition; if instead the observer's record is an ordinary
  classical fact, that is a hidden-variable model and the ceiling of $2$ applies to it.
- **Teleportation** moves an unknown state with a shared Bell pair plus two classical
  bits: `CNOT`, `H`, measure, then `X`/`Z` corrections chosen by a plain Python `if`. The
  original is destroyed — Alice's qubits end at Bloch $z = \pm1$ — so no-cloning is
  respected, and the phone call is why nothing outruns light.
- **Superdense coding** is the reverse trade: a shared Bell pair plus one transmitted
  qubit carries two classical bits, because Alice can steer the pair among the four
  orthogonal Bell states by acting on her half alone.

New library surface from this notebook: `qsim.algorithms.chsh` (`chsh_expectation`,
`chsh_S`, `chsh_sampled`, `classical_bound`, `classical_strategies`, `OPTIMAL_SETTINGS`,
`CLASSICAL_LIMIT`, `QUANTUM_LIMIT`) and `qsim.algorithms.teleportation` (`teleport`,
`superdense_send`).

## Next: notebook 04

Everything so far has been written gate by gate, by hand. That stops scaling immediately:
the algorithms in the second half of this project need blocks of gates that can be made
**controlled** (run this whole subroutine only if that qubit is $1$) and **reversed** (run
that whole subroutine backwards), plus scratch qubits that can be borrowed and returned
clean.

`04-combinators.ipynb` builds those tools. It is the most software-engineering-flavoured
notebook in the series, but the physics keeps intruding in useful ways — in particular,
you will find out why a borrowed scratch qubit that is not returned *exactly* to
$|0\rangle$ silently destroys the interference every later algorithm depends on, and why
qsim raises `DirtyAncillaError` rather than letting you find out the hard way.